# 🧠 NER using BiLSTM (Simple + Realistic)

## ✅ What this covers

* Tokenization
* Label encoding
* Padding sequences
* BiLSTM model
* Prediction


# 📦 Step 1: Import libraries

In [5]:
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Bidirectional, Dense, TimeDistributed
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical


# 📊 Step 2: Create sample dataset

We use **BIO tagging format**:

* B-PER → Beginning of Person
* I-PER → Inside Person
* O → Outside

In [10]:
sentences = [
    ["John", "lives", "in", "New", "York"],
    ["Mary", "works", "at", "Google"],
    ["Rahul", "is", "from", "India"],
    ["Alice", "is", "from", "London"],
    ["Bob", "works", "at", "Microsoft"],
    ["Google", "is", "in", "USA"],
    ["Amazon", "is", "a", "company"],
    ["Elon", "Musk", "founded", "Tesla"],
    ["Sundar", "Pichai", "works", "at", "Google"],
    ["Delhi", "is", "in", "India"]
]

tags = [
    ["B-PER", "O", "O", "B-LOC", "I-LOC"],
    ["B-PER", "O", "O", "B-ORG"],
    ["B-PER", "O", "O", "B-LOC"],
    ["B-PER", "O", "O", "B-LOC"],
    ["B-PER", "O", "O", "B-ORG"],
    ["B-ORG", "O", "O", "B-LOC"],
    ["B-ORG", "O", "O", "O"],
    ["B-PER", "I-PER", "O", "B-ORG"],
    ["B-PER", "I-PER", "O", "O", "B-ORG"],
    ["B-LOC", "O", "O", "B-LOC"]
]


# 🔤 Step 3: Create vocab and mappings

In [11]:
words = list(set([w for s in sentences for w in s]))
words.append("PAD")

word2idx = {w: i+1 for i, w in enumerate(words)}

tag_set = list(set([t for tag_list in tags for t in tag_list]))
tag2idx = {t: i for i, t in enumerate(tag_set)}


# 🔢 Step 4: Convert to sequences

In [12]:
X = [[word2idx[w] for w in s] for s in sentences]
y = [[tag2idx[t] for t in tag_list] for tag_list in tags]


# 📏 Step 5: Padding

In [13]:
max_len = max(len(s) for s in sentences)

X = pad_sequences(X, maxlen=max_len, padding="post")
y = pad_sequences(y, maxlen=max_len, padding="post")

y = [to_categorical(i, num_classes=len(tag2idx)) for i in y]


### Why padding?

* Neural networks need **fixed input length**



# 🧠 Step 6: Build BiLSTM Model

In [14]:
input_layer = Input(shape=(max_len,))

embedding = Embedding(input_dim=len(word2idx)+1, output_dim=128)(input_layer)
bilstm = Bidirectional(LSTM(units=64, return_sequences=True))(embedding)


output = TimeDistributed(Dense(len(tag2idx), activation="softmax"))(bilstm)

model = Model(input_layer, output)

model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()


Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 5)]               0         
                                                                 
 embedding_1 (Embedding)     (None, 5, 128)            3840      
                                                                 
 bidirectional_1 (Bidirecti  (None, 5, 128)            98816     
 onal)                                                           
                                                                 
 time_distributed_1 (TimeDi  (None, 5, 6)              774       
 stributed)                                                      
                                                                 
Total params: 103430 (404.02 KB)
Trainable params: 103430 (404.02 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


### Why BiLSTM?

* Looks **left + right context**
* Important for NER
  👉 “Apple” could be company or fruit → context matters



###  Why TimeDistributed?

* We predict **label for each word**, not whole sentence


# 🚀 Step 7: Train model

In [15]:
# model.fit(X, np.array(y), batch_size=2, epochs=10) # gives poor result
model.fit(X, np.array(y), batch_size=2, epochs=50)


Epoch 1/50
5/5 [==============================] - 4s 17ms/step - loss: 1.7807 - accuracy: 0.4600
Epoch 2/50
5/5 [==============================] - 0s 7ms/step - loss: 1.7426 - accuracy: 0.5600
Epoch 3/50
5/5 [==============================] - 0s 10ms/step - loss: 1.6977 - accuracy: 0.5600
Epoch 4/50
5/5 [==============================] - 0s 6ms/step - loss: 1.6435 - accuracy: 0.5600
Epoch 5/50
5/5 [==============================] - 0s 6ms/step - loss: 1.5720 - accuracy: 0.5600
Epoch 6/50
5/5 [==============================] - 0s 6ms/step - loss: 1.4825 - accuracy: 0.5600
Epoch 7/50
5/5 [==============================] - 0s 5ms/step - loss: 1.3651 - accuracy: 0.5600
Epoch 8/50
5/5 [==============================] - 0s 5ms/step - loss: 1.2611 - accuracy: 0.5600
Epoch 9/50
5/5 [==============================] - 0s 5ms/step - loss: 1.1525 - accuracy: 0.5600
Epoch 10/50
5/5 [==============================] - 0s 5ms/step - loss: 1.0422 - accuracy: 0.6200
Epoch 11/50
5/5 [====================

# 🔍 Step 8: Prediction

In [16]:
idx2tag = {i: t for t, i in tag2idx.items()}

test_sentence = ["John", "works", "in", "India"]

test_seq = [word2idx.get(w, word2idx["PAD"]) for w in test_sentence]

test_seq = pad_sequences([test_seq], maxlen=max_len, padding="post")

pred = model.predict(test_seq)

pred_labels = np.argmax(pred, axis=-1)[0]

for word, label in zip(test_sentence, pred_labels):
    print(word, "->", idx2tag[label])


1/1 [==============================] - 1s 1s/step
John -> B-PER
works -> O
in -> O
India -> B-LOC



### Limitations (important for realism)

* Small dataset → poor generalization
* No embeddings like Word2Vec/GloVe
* No CRF layer (used in real systems)

---

# 🚀 Upgrade 


> “Real production models use **BiLSTM + CRF** or Transformers like BERT”

---
